<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [52]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

In [51]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
def load_data(seed=42):
    """
    Carrega o Fashion MNIST via fetch_openml e realiza
    a separação treino/teste com controle de aleatoriedade.
 
    Parâmetros:
        seed : int
            Semente para reprodutibilidade do train_test_split.
 
    Retorna:
        X_train, X_test, y_train, y_test : arrays numpy
    """
    print("Carregando Fashion MNIST")
    dataset = fetch_openml("Fashion-MNIST", version=1, as_frame=False, parser="auto")
    X, y = dataset.data, dataset.target.astype(int)
 
    # Normalização simples (0-255 -> 0-1), isso é uma boa prática para modelos baseados em árvore
    X = X / 255.0
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
 
    print(f"Treino : {X_train.shape[0]} amostras")
    print(f"Teste  : {X_test.shape[0]} amostras")
    print(f"Features: {X_train.shape[1]}")
    return X_train, X_test, y_train, y_test

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [50]:
def train_random_forest(X_train, y_train, seed=42):
    """
    Treina um Random Forest com random_state fixo para reprodutibilidade.
 
    Parâmetros
        X_train: array-like
        y_train: array-like
        seed: int

    Retorna:
        model: RandomForestClassifier treinado
    """
    print("\nTreinando Random Forest...")
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1          
    )
    model.fit(X_train, y_train)
    return model
 
 
def train_adaboost(X_train, y_train, seed=42):
    """
    Treina um AdaBoost com random_state fixo para reprodutibilidade.
 
    Parâmetros:
        X_train: array-like
        y_train: array-like
        seed: int
 
    Retorna:
        model: AdaBoostClassifier treinado
    """
    model = AdaBoostClassifier(
        n_estimators=100,
        random_state=seed,
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [49]:
from sklearn.metrics import accuracy_score
def evaluate(model, X_test, y_test):
    """
    Realiza predições e retorna a acurácia do modelo.
 
    Parâmetros
        model: estimador sklearn treinado
        X_test: array-like
        y_test: array-like
 
    Retorna
        accuracy: float
    """
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy


**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [53]:
def run_pipeline(model_type="rf", seed=42):
    """
    Executa o pipeline completo: carga -> treino -> avaliação.
 
    Parâmetros:
        model_type: str "rf" para Random Forest | "ab" para AdaBoost
        seed: int Semente para reprodutibilidade
 
    Retorna
        accuracy: float
    """
    X_train, X_test, y_train, y_test = load_data(seed)
 
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError(f"model_type '{model_type}' inválido. Use 'rf' ou 'ab'.")
 
    accuracy = evaluate(model, X_test, y_test)
    return accuracy

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [54]:
seed = 42

def evaluate_full(model, X_test, y_test):
    """Retorna (acurácia, precisão, recall, f1) com média ponderada."""
    y_pred = model.predict(X_test)
    return (
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted"),
    )
 
 
X_train, X_test, y_train, y_test = load_data(seed)
 
rf = train_random_forest(X_train, y_train, seed)
ab = train_adaboost(X_train, y_train, seed)
 
rf_metrics = evaluate_full(rf, X_test, y_test)
ab_metrics = evaluate_full(ab, X_test, y_test)
 
print("Random Forest")
print(f"Acurácia: {rf_metrics[0]:.4f}")
print(f"Precisão: {rf_metrics[1]:.4f}")
print(f"Recall: {rf_metrics[2]:.4f}")
print(f"F1-score: {rf_metrics[3]:.4f}")
 
print("\nAdaBoost")
print(f"Acurácia: {ab_metrics[0]:.4f}")
print(f"Precisão: {ab_metrics[1]:.4f}")
print(f"Recall: {ab_metrics[2]:.4f}")
print(f"F1-score: {ab_metrics[3]:.4f}")

Carregando Fashion MNIST
Treino : 56000 amostras
Teste  : 14000 amostras
Features: 784

Treinando Random Forest...
Random Forest
Acurácia: 0.8819
Precisão: 0.8807
Recall: 0.8819
F1-score: 0.8800

AdaBoost
Acurácia: 0.5729
Precisão: 0.6115
Recall: 0.5729
F1-score: 0.5532


addBoost pode ate ser mais rapido, porem sua simplicidade que faz ele ser rapido é uma facada de dois gumes deixando incapaz de aprender como a robustez do random forst pois ele consegue criar varios modelos deixando sua especialidade para cada parte ja que os dados sao aleatorios e sua profundidade nao se limita a 1

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
seed = 7

def evaluate_full(model, X_test, y_test):
    """Retorna (acurácia, precisão, recall, f1) com média ponderada."""
    y_pred = model.predict(X_test)
    return (
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="weighted"),
        recall_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, y_pred, average="weighted"),
    )
 
 
X_train, X_test, y_train, y_test = load_data(seed)
 
rf = train_random_forest(X_train, y_train, seed)
ab = train_adaboost(X_train, y_train, seed)
 
rf_metrics = evaluate_full(rf, X_test, y_test)
ab_metrics = evaluate_full(ab, X_test, y_test)
 
print("Random Forest")
print(f"Acurácia: {rf_metrics[0]:.4f}")
print(f"Precisão: {rf_metrics[1]:.4f}")
print(f"Recall: {rf_metrics[2]:.4f}")
print(f"F1-score: {rf_metrics[3]:.4f}")
 
print("\nAdaBoost")
print(f"Acurácia: {ab_metrics[0]:.4f}")
print(f"Precisão: {ab_metrics[1]:.4f}")
print(f"Recall: {ab_metrics[2]:.4f}")
print(f"F1-score: {ab_metrics[3]:.4f}")

Carregando Fashion MNIST
Treino : 56000 amostras
Teste  : 14000 amostras
Features: 784

Treinando Random Forest...


Sim, os resultados mudam devido a como os dados sao distribuidos e organizados para treinar o modelo 

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [18]:
# TODO: implemente load_data

é mais comum  no random forst por nao ter um limite de profundidade como o addbost

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [19]:
# TODO: implemente load_data

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [20]:
# TODO: implemente load_data